# Feature Store: Manage Features and Generate Training Data

This notebook demonstrates the `snowflakeR` package interface to the Snowflake Feature Store.
You'll learn how to define entities, create feature views, generate training data with
point-in-time correct joins, and tie it all together with the Model Registry.

This notebook is for **Snowflake Workspace Notebooks** (Python kernel + `%%R` magic).
For local environments (RStudio, Posit, JupyterLab), use `local_feature_store.ipynb`.

**Before you start:** Optionally edit `snowflaker_feature_store_config.yaml` to set your
database, schema, warehouse, and EAI settings.

**Sections:**
1. Setup
2. Connect & Feature Store Context
3. Entities
4. Feature Views
5. Training Data Generation
6. Retrieve Features for Inference
7. Time Aggregation Features
8. Online Feature Serving
9. End-to-End: Feature Store + Model Registry
10. Cleanup


In [ ]:
import time as _time
_nb_start = _time.time()
print(f"Notebook started: {_time.strftime('%Y-%m-%d %H:%M:%S')}")

## 1. Setup

Single cell to set session context, validate EAI, install R, and install packages.

In [ ]:
from sfnb_setup import setup_notebook
setup_notebook(config="snowflaker_feature_store_config.yaml")

## 2. Connect & Feature Store Context

Session context was set by `setup_notebook()` above.
All table references in this notebook use fully qualified names via `sfr_fqn()`.

In [ ]:
%%R
library(snowflakeR)

# Connect (auto-detects Workspace session; context set by setup_notebook)
conn <- sfr_connect()
conn

In [ ]:
%%R
# Create a Feature Store context targeting the configured schema
# `create = TRUE` creates the schema and required tags if they don't exist
fs <- sfr_feature_store(
  conn,
  database  = conn$database,
  schema    = conn$schema,
  warehouse = conn$warehouse,
  create    = TRUE
)

fs

### What is an `sfr_feature_store` object?

It holds the connection and the target database/schema/warehouse for all Feature Store
operations. Pass it as the first argument to all `sfr_*_entity()` and `sfr_*_feature_view()` functions.

---

## 3. Entities

**Entities** define join keys that link features to business objects (customers, products, etc.).
They are the foundation for Feature Views.

### Create sample data

First, let's create some sample tables to work with.

In [ ]:
%%R
# Create sample order data
sfr_execute(conn, paste(
  "CREATE OR REPLACE TABLE", sfr_fqn(conn, "SFR_DEMO_ORDERS"), "(",
  "  customer_id INT,",
  "  order_date  DATE,",
  "  order_total DOUBLE",
  ")"
))

sfr_execute(conn, paste(
  "INSERT INTO", sfr_fqn(conn, "SFR_DEMO_ORDERS"), "VALUES",
  "(1, '2025-01-15', 45.50),",
  "(1, '2025-02-20', 82.30),",
  "(1, '2025-03-10', 15.00),",
  "(2, '2025-01-22', 120.00),",
  "(2, '2025-03-05', 55.75),",
  "(3, '2025-02-01', 200.00),",
  "(3, '2025-02-15', 30.25),",
  "(3, '2025-03-20', 95.50)"
))

# Create sample label data
sfr_execute(conn, paste(
  "CREATE OR REPLACE TABLE", sfr_fqn(conn, "SFR_DEMO_LABELS"), "(",
  "  customer_id INT,",
  "  churned     INT",
  ")"
))

sfr_execute(conn, paste(
  "INSERT INTO", sfr_fqn(conn, "SFR_DEMO_LABELS"),
  "VALUES (1, 0), (2, 1), (3, 0)"
))

rcat("Sample tables created.")

### Create and manage entities

In [ ]:
%%R
# Create a customer entity
customer <- sfr_create_entity(
  fs,
  name      = "SFR_DEMO_CUSTOMER",
  join_keys = "CUSTOMER_ID",
  desc      = "Demo customer entity"
)

customer

In [ ]:
%%R
# List all entities
entities <- sfr_list_entities(fs)
entities

In [ ]:
%%R
# Get a specific entity
customer <- sfr_get_entity(fs, "SFR_DEMO_CUSTOMER")
customer

In [ ]:
%%R
# Update the description
sfr_update_entity(fs, "SFR_DEMO_CUSTOMER", desc = "Primary customer entity for demo")

---

## 4. Feature Views

**Feature Views** define the SQL transformation that produces features.
They can be:
- **Managed:** Automatically refreshed as a dynamic table (`refresh_freq` specified)
- **External:** Manually maintained (no `refresh_freq`)

### Create a Feature View from SQL

In [ ]:
%%R
# One-step creation: SQL-based Feature View
fv <- sfr_create_feature_view(
  fs,
  name     = "SFR_DEMO_CUST_FEATURES",
  version  = "v1",
  entities = customer,
  feature_df = paste("
    SELECT
      customer_id,
      AVG(order_total)   AS avg_order_total,
      COUNT(*)::INT      AS order_count,
      SUM(order_total)   AS total_spend,
      MAX(order_date)    AS last_order_date
    FROM", sfr_fqn(conn, "SFR_DEMO_ORDERS"), "
    GROUP BY customer_id
  "),
  desc = "Customer aggregate features from orders",
  overwrite = TRUE
)

fv

In [ ]:
%%R
# Alternative: dbplyr-based Feature View
# The features argument accepts a dbplyr lazy table -- extract_feature_sql()
# converts it to SQL automatically via dbplyr::sql_render().
# Requires RSnowflake for the DBI connection that dbplyr needs.

if (requireNamespace("RSnowflake", quietly = TRUE) &&
    requireNamespace("dbplyr", quietly = TRUE) &&
    requireNamespace("dplyr", quietly = TRUE)) {

  library(dplyr)
  library(dbplyr)

  dbi_con <- sfr_dbi_connection(conn)
  orders_tbl <- tbl(dbi_con, I(sfr_fqn(conn, "SFR_DEMO_ORDERS")))

  # Build the same aggregation as the SQL example above, using dplyr
  features_query <- orders_tbl |>
    group_by(CUSTOMER_ID) |>
    summarise(
      avg_order_total = mean(ORDER_TOTAL, na.rm = TRUE),
      order_count     = n(),
      total_spend     = sum(ORDER_TOTAL, na.rm = TRUE),
      last_order_date = max(ORDER_DATE, na.rm = TRUE)
    )

  # Pass the dbplyr lazy table directly as features
  fv_dbplyr <- sfr_create_feature_view(
    fs,
    name     = "SFR_DEMO_CUST_FEATURES_DBPLYR",
    version  = "v1",
    entities = customer,
    feature_df = features_query,
    desc     = "Same customer features, defined via dplyr instead of SQL"
  )

  fv_dbplyr
} else {
  rcat("Skipping dbplyr Feature View example (RSnowflake not installed).")
}

### Alternative: Two-step (draft then register)

This mirrors the Python API and is useful when you want to inspect the draft:

```r
%%R
# Step 1: Create a local draft
fv_draft <- sfr_feature_view(
  name     = "MY_FEATURES",
  entities = customer,
  feature_df = "SELECT ... FROM ...",
  refresh_freq = "1 hour"
)

# Step 2: Register (materialise)
fv <- sfr_register_feature_view(fs, fv_draft, version = "v1")
```

### Alternative: dbplyr-based features

Use `sfr_dbi_connection()` to get an RSnowflake DBI connection for dbplyr:

```r
%%R
library(dplyr); library(dbplyr)

dbi_con <- sfr_dbi_connection(conn)
orders_tbl <- tbl(dbi_con, I(sfr_fqn(conn, "SFR_DEMO_ORDERS")))
features_query <- orders_tbl |>
  group_by(CUSTOMER_ID) |>
  summarise(avg_total = mean(ORDER_TOTAL), order_count = n())

fv <- sfr_create_feature_view(
  fs, "CUST_FV_DBPLYR", "v1",
  entities = customer,
  feature_df = features_query   # dbplyr lazy table -> SQL
)
```

### Manage Feature Views

In [ ]:
%%R
# List all Feature Views
fvs <- sfr_list_feature_views(fs)
fvs

In [ ]:
%%R
# Get a specific version
fv <- sfr_get_feature_view(fs, "SFR_DEMO_CUST_FEATURES", "v1")
fv

In [ ]:
%%R
# Read feature data directly
feature_data <- sfr_read_feature_view(fs, "SFR_DEMO_CUST_FEATURES", "v1")
feature_data

### Refresh management (for managed Feature Views)

```r
%%R
# Manually trigger a refresh
sfr_refresh_feature_view(fs, "MY_FV", "v1")

# Check refresh history
sfr_get_refresh_history(fs, "MY_FV", "v1")

# Pause/resume automatic refresh
sfr_suspend_feature_view(fs, "MY_FV", "v1")
sfr_resume_feature_view(fs, "MY_FV", "v1")
```

---

## 5. Training Data Generation

Join **spine** (label) data with Feature Views using point-in-time correct joins.
This ensures no data leakage -- features are joined as-of the label timestamp.

`sfr_generate_training_data()` maps to the Python SDK's `fs.generate_training_set()`,
which returns an ephemeral Snowpark DataFrame. For an immutable, versioned Dataset
with ML Lineage support (Feature View -> Dataset -> Model), use
`sfr_generate_dataset()` instead -- demonstrated in Section 9.

In [ ]:
%%R
# Generate training data by joining labels with features
training_data <- sfr_generate_training_data(
  fs,
  spine = paste("SELECT customer_id, churned FROM", sfr_fqn(conn, "SFR_DEMO_LABELS")),
  features = list(
    list(name = "SFR_DEMO_CUST_FEATURES", version = "v1")
  ),
  spine_label_cols = "CHURNED"
)

training_data

The result is a regular R data.frame -- ready for `lm()`, `glm()`, `randomForest()`, etc.

---

## 6. Retrieve Features for Inference

`sfr_retrieve_features()` maps to the Python SDK's `fs.retrieve_feature_values()`.

- **Without `spine_timestamp_col`** (the default, shown below): performs a LEFT JOIN
  on entity keys and returns the **current** feature values from the Feature View.
- **With `spine_timestamp_col`**: performs the same point-in-time join as
  `sfr_generate_training_data()`, returning feature values as-of each spine timestamp.

In [ ]:
%%R
# Get current features for all customers
inference_features <- sfr_retrieve_features(
  fs,
  spine = paste("SELECT DISTINCT customer_id FROM", sfr_fqn(conn, "SFR_DEMO_ORDERS")),
  features = list(
    list(name = "SFR_DEMO_CUST_FEATURES", version = "v1")
  )
)

inference_features

---

## 7. Time Aggregation Features

Snowflake Feature Store supports **tile-based time aggregation** that automatically
computes rolling window metrics (SUM, AVG, COUNT, MIN, MAX) over streaming data.
Instead of writing complex SQL window functions, you define `sfr_feature()` objects
and a `feature_granularity`, and the Feature Store handles the rest.

This requires `snowflake-ml-python >= 1.24.0`.


In [ ]:
%%R
# Check version compatibility
sfr_ml_version()


In [ ]:
%%R
# Create a timestamped transaction stream for aggregation
sfr_execute(conn, paste(
  "CREATE OR REPLACE TABLE", sfr_fqn(conn, "SFR_DEMO_TRANSACTIONS"), "(",
  "  customer_id INT,",
  "  amount      DOUBLE,",
  "  category    VARCHAR,",
  "  event_time  TIMESTAMP_NTZ",
  ")"
))

sfr_execute(conn, paste(
  "INSERT INTO", sfr_fqn(conn, "SFR_DEMO_TRANSACTIONS"), "VALUES",
  "(1, 45.50, 'electronics', '2025-01-15 10:30:00'),",
  "(1, 82.30, 'clothing',    '2025-02-20 14:15:00'),",
  "(1, 15.00, 'groceries',   '2025-03-10 09:00:00'),",
  "(2, 120.00,'electronics', '2025-01-22 11:45:00'),",
  "(2, 55.75, 'clothing',    '2025-03-05 16:30:00'),",
  "(3, 200.00,'electronics', '2025-02-01 08:00:00'),",
  "(3, 30.25, 'groceries',   '2025-02-15 12:00:00'),",
  "(3, 95.50, 'clothing',    '2025-03-20 15:45:00')"
))

rcat("Transaction stream table created.")


In [ ]:
%%R
# Define aggregation features: 30-day rolling windows over the transaction stream
agg_features <- list(
  sfr_feature("AMOUNT", "FLOAT", "SUM",   "30 days"),
  sfr_feature("AMOUNT", "FLOAT", "AVG",   "30 days"),
  sfr_feature("AMOUNT", "FLOAT", "COUNT", "30 days")
)

# Create a Feature View with tile-based aggregation
# feature_granularity controls the tile size (how often aggregates are recomputed)
agg_fv <- sfr_create_feature_view(
  fs,
  name       = "SFR_DEMO_TXN_AGG",
  version    = "v1",
  entities   = customer,
  feature_df = paste(
    "SELECT customer_id, amount, event_time FROM",
    sfr_fqn(conn, "SFR_DEMO_TRANSACTIONS")
  ),
  timestamp_col       = "EVENT_TIME",
  features            = agg_features,
  feature_granularity = "1 day",
  refresh_freq        = "1 hour",
  desc = "30-day rolling transaction aggregates (tile-based)",
  overwrite = TRUE
)

agg_fv


In [ ]:
%%R
# Read the aggregated features
agg_data <- sfr_read_feature_view(fs, "SFR_DEMO_TXN_AGG", "v1")
agg_data


### When to use tile-based aggregation vs. plain SQL

| Approach | Best for | Trade-off |
|----------|----------|-----------|
| **Tile-based** (`sfr_feature()`) | Time-windowed features over data that is sparse on the timestamp grain; prevents shifted windows when source event timestamps don't align with the spine ASOF date | Requires `feature_granularity`; FS manages the compute |
| **SQL windowing** (`feature_df` with `RANGE BETWEEN`) | Simple one-off aggregations, full control over SQL | You manage refresh and correctness; risk of shifted time-windows when source data is sparse |

Use `sfr_feature()` whenever time-windowed features are needed and the source data
is sparse on the timestamp grain. Without tile-based aggregation, a PIT join could
return a feature value computed from an earlier time-window than the ASOF date in
the spine, resulting in an incorrect/shifted window.

Use plain SQL in `feature_df` for simpler, ad-hoc feature engineering where
timestamp alignment is not a concern.


---

## 8. Online Feature Serving

Feature Views can be configured for **online serving** -- low-latency key lookups
backed by Snowflake hybrid tables. This is useful for real-time inference where you
need features for specific entities (e.g., a single customer) rather than batch reads.

This requires `snowflake-ml-python >= 1.18.0` and the `CREATE ONLINE FEATURE TABLE`
privilege.


In [ ]:
%%R
# Create an online configuration
# target_lag controls how fresh the online copy is relative to the offline source
online_cfg <- sfr_online_config(enable = TRUE, target_lag = "1 minute")
online_cfg


In [ ]:
%%R
# Drop any stale online table from a previous run to avoid type-mismatch
# errors when the Feature View was recreated.
tryCatch(
  sfr_execute(conn, paste(
    'DROP ONLINE FEATURE TABLE IF EXISTS',
    sfr_fqn(conn, '"SFR_DEMO_CUST_FEATURES$v1$ONLINE"')
  )),
  error = function(e) rcat("  (no stale online table to drop)")
)

# Apply online serving to the basic customer features FV
sfr_update_feature_view(
  fs,
  name           = "SFR_DEMO_CUST_FEATURES",
  version        = "v1",
  online_config  = online_cfg
)

rcat("Online serving enabled for SFR_DEMO_CUST_FEATURES v1")


In [ ]:
%%R
# Read features from the online store (keyed lookup for specific customers)
# Contrast with the offline read in Section 6 which returns all rows.
# The online table may take a minute to populate after enabling --
# poll with backoff until it is ready.

max_attempts <- 6
wait_secs    <- 10
online_features <- NULL

for (attempt in seq_len(max_attempts)) {
  online_features <- tryCatch(
    sfr_read_feature_view(
      fs, "SFR_DEMO_CUST_FEATURES", "v1",
      store_type    = "ONLINE",
      keys          = list(CUSTOMER_ID = c(1L, 3L)),
      feature_names = c("AVG_ORDER_TOTAL", "ORDER_COUNT")
    ),
    error = function(e) {
      if (grepl("not refreshed yet", conditionMessage(e))) {
        rcat(sprintf("Attempt %d/%d: online table not ready, retrying in %ds...",
                     attempt, max_attempts, wait_secs))
        Sys.sleep(wait_secs)
        return(NULL)
      }
      stop(e)
    }
  )
  if (!is.null(online_features)) break
}

if (is.null(online_features)) {
  stop("Online feature table did not become ready within ",
       max_attempts * wait_secs, "s. Try re-running this cell.")
}

rcat("Online features (2 customers, 2 features):")
online_features


### Data flow: source table -> Feature View -> Online table

The values above are the **before** state. Now insert new rows into the source
table and observe them propagate through the Feature View (Dynamic Table) into
the Online table.

In [ ]:
%%R
# Insert new orders for customer 1 to change the aggregate features
sfr_execute(conn, paste(
  "INSERT INTO", sfr_fqn(conn, "SFR_DEMO_ORDERS"), "VALUES",
  "(1, '2025-04-01', 500.00),",
  "(1, '2025-04-05', 250.00)"
))

rcat("Inserted 2 new orders for customer 1 (total +750).")
rcat("Before: avg ~47.6, count=3, total=142.8")
rcat("Expected after: avg ~178.6, count=5, total=892.8")


In [ ]:
%%R
# Wait for the Feature View DT to refresh and the online table to sync.
# Total expected lag = DT refresh interval + online target_lag.
# Our FV has no explicit refresh_freq (external), and target_lag = 1 minute,
# so we poll until the values change or a timeout is reached.

rcat("Waiting for data to propagate (polling every 15s, up to 3 min)...")
deadline <- Sys.time() + 180
updated <- FALSE

while (Sys.time() < deadline) {
  Sys.sleep(15)
  after <- sfr_read_feature_view(
    fs, "SFR_DEMO_CUST_FEATURES", "v1",
    store_type    = "ONLINE",
    keys          = list(CUSTOMER_ID = 1L),
    feature_names = c("AVG_ORDER_TOTAL", "ORDER_COUNT", "TOTAL_SPEND")
  )
  rcat(paste("  ORDER_COUNT =", after$ORDER_COUNT))
  if (after$ORDER_COUNT >= 5) {
    updated <- TRUE
    break
  }
}

if (updated) {
  rcat("Online table updated -- new feature values for customer 1:")
  after
} else {
  rcat("Timeout -- DT may need a manual refresh (external FV) or longer lag.")
  rcat("Try: sfr_refresh_feature_view(fs, 'SFR_DEMO_CUST_FEATURES', 'v1')")
}


### Understanding propagation lag

End-to-end lag from a source table change to the online table has two components:

1. **Feature View DT refresh** -- controlled by `refresh_freq` when creating the
   Feature View. For managed Feature Views this is the Dynamic Table refresh interval.
   For external Feature Views you must refresh manually or on a schedule.
2. **Online table sync** -- controlled by `target_lag` in the online config.

Total expected lag = `refresh_freq` + `target_lag`.

To measure actual propagation time: compare the `SYSTEM$STREAM_HAS_DATA` or DT
refresh history timestamps against the online table's `METADATA$ROW_VERSION`.


In [ ]:
%%R
# Disable online serving when done
sfr_update_feature_view(
  fs,
  name          = "SFR_DEMO_CUST_FEATURES",
  version       = "v1",
  online_config = sfr_online_config(enable = FALSE)
)

rcat("Online serving disabled.")


### Online serving considerations

- **Latency:** `target_lag` sets how fresh the online table is. Lower lag = more compute cost.
- **Privileges:** Requires `CREATE ONLINE FEATURE TABLE` on the schema.
- **Cost:** Online tables are backed by hybrid tables which have separate storage costs.
- **Use case:** Real-time serving (e.g., API endpoints, streaming pipelines). For batch
  scoring, the standard offline read is sufficient and cheaper.

### Hybrid table warm-up

Hybrid tables require a few minutes of concurrent queries (hundreds to thousands) to
**warm up** before reaching steady-state operational performance. When benchmarking:

- Run with a **production-level query concurrency** for sufficient duration.
- Measure **aggregate response times per minute** throughout the test.
- Do not draw conclusions from cold-start latency -- performance improves as the
  table reaches its operational level.


---

## 9. End-to-End -- Feature Store + Model Registry

Tie everything together: generate training data from the Feature Store,
train a model in R, register it, and score new customers.

This section uses `sfr_generate_dataset()` (the immutable Dataset API) instead of
`sfr_generate_training_data()` (shown in Section 5). The Dataset enables
**ML Lineage**: Feature View -> Dataset -> Model. Lineage currently requires
the Dataset-based path via `sfr_generate_dataset()`.

In [ ]:
%%R
# 1. Generate an immutable Dataset from Feature Store (enables ML Lineage)
training <- sfr_generate_dataset(
  fs,
  name = "SFR_DEMO_CHURN_TRAINING",
  spine = paste("SELECT customer_id, churned FROM", sfr_fqn(conn, "SFR_DEMO_LABELS")),
  features = list(
    list(name = "SFR_DEMO_CUST_FEATURES", version = "v1")
  ),
  version = "v1",
  spine_label_cols = "CHURNED"
)

rcat("Training data (Dataset):")
str(training)

In [ ]:
%%R
# 2. Train a model in R
model <- glm(
  CHURNED ~ AVG_ORDER_TOTAL + ORDER_COUNT + TOTAL_SPEND,
  data   = training,
  family = binomial
)

summary(model)

In [ ]:
%%R
# 3. Test locally
test_input <- training[, c("AVG_ORDER_TOTAL", "ORDER_COUNT", "TOTAL_SPEND")]
preds <- sfr_predict_local(model, test_input)
cbind(training[, c("CUSTOMER_ID", "CHURNED")], preds)

In [ ]:
%%R
# 4. Register to Model Registry with Dataset lineage
reg <- sfr_model_registry(conn)

mv <- sfr_log_model(
  reg,
  model      = model,
  model_name = "SFR_DEMO_CHURN",
  input_cols = list(
    AVG_ORDER_TOTAL = "double",
    ORDER_COUNT     = "double",
    TOTAL_SPEND     = "double"
  ),
  output_cols = list(prediction = "double"),
  training_dataset = training,
  comment = "Logistic regression for customer churn (with FS lineage)"
)

mv

In [ ]:
%%R
# 5. Score new customers using Feature Store features
new_features <- sfr_retrieve_features(
  fs,
  spine = paste("SELECT DISTINCT customer_id FROM", sfr_fqn(conn, "SFR_DEMO_ORDERS")),
  features = list(
    list(name = "SFR_DEMO_CUST_FEATURES", version = "v1")
  )
)

# Local prediction (or use sfr_predict for remote)
scores <- sfr_predict_local(
  model,
  new_features[, c("AVG_ORDER_TOTAL", "ORDER_COUNT", "TOTAL_SPEND")]
)

cbind(new_features[, "CUSTOMER_ID", drop = FALSE], churn_score = scores$prediction)

---

## 10. Cleanup

In [ ]:
%%R
# Uncomment to clean up demo objects
# (commented out to avoid accidental deletion on Run All)
#
# sfr_delete_model(reg, "SFR_DEMO_CHURN")
# sfr_execute(conn, paste("DROP DATASET IF EXISTS", sfr_fqn(conn, "SFR_DEMO_CHURN_TRAINING")))
# sfr_delete_feature_view(fs, "SFR_DEMO_CUST_FEATURES", "v1")
# sfr_delete_feature_view(fs, "SFR_DEMO_TXN_AGG", "v1")
# sfr_delete_entity(fs, "SFR_DEMO_CUSTOMER")
#
# sfr_execute(conn, paste("DROP TABLE IF EXISTS", sfr_fqn(conn, "SFR_DEMO_TRANSACTIONS")))
# sfr_execute(conn, paste("DROP TABLE IF EXISTS", sfr_fqn(conn, "SFR_DEMO_ORDERS")))
# sfr_execute(conn, paste("DROP TABLE IF EXISTS", sfr_fqn(conn, "SFR_DEMO_LABELS")))
#
# sfr_disconnect(conn)
# rcat("All demo objects cleaned up.")

---

## Next steps

- **Model Registry:** See `workspace_model_registry.ipynb` for training, registration,
  and experiment tracking.
- **Model Consumption:** See `workspace_model_consumption.ipynb` for SQL-direct
  batch inference and cross-language models.
- **Model Monitoring:** See `workspace_model_monitoring.ipynb` for drift detection
  and performance tracking.

In [ ]:
_nb_elapsed = _time.time() - _nb_start
_mins, _secs = divmod(_nb_elapsed, 60)
print(f"Notebook completed: {_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total execution time: {int(_mins)}m {_secs:.1f}s")

In [ ]:
# CI results -- writes a results table when run non-interactively
try:
    from snowflake.snowpark.context import get_active_session
    _ci_session = get_active_session()
    _ci_session.sql("""
        CREATE OR REPLACE TABLE NOTEBOOK_CI.FEATURE_STORE_RESULTS AS
        SELECT 'feature_store' AS TEST_NAME, 'PASS' AS STATUS,
               CURRENT_TIMESTAMP() AS RUN_AT, CURRENT_USER() AS RUN_BY
    """).collect()
    print("CI results written to NOTEBOOK_CI.FEATURE_STORE_RESULTS")
except Exception:
    pass